In [2]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from two_level_mc import * # adjust if your notebook is nested differently
from functions_v2 import *

In [5]:
edges, n_vertices, weights = load_graph(r"../../data/raw/facebook_combined.txt")
print(f"n_vertices={n_vertices}, edges={len(edges)}")

✓ Loaded: ../../data/raw/facebook_combined.txt
  Vertices : 4039
  Edges    : 88234
  Weighted : no

n_vertices=4039, edges=88234


In [8]:
check_connectivity(edges, n_vertices)

Components: 1
  -> Graph is fully connected, safe to proceed



1

In [11]:
# 3. Phase 1
A, D, L = build_graph_matrices(edges, n_vertices)

# 4. Phase 2 — sparse from the start, given graph size
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)
print("L_sigma type:", type(L_sigma))

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4039 x 4039
  Degree range: [1, 1045]
  Non-zeros in L: 180507

✓ lambda_min = 0.000837
  (eigenvalues found: [0.         0.00083651])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

L_sigma type: <class 'scipy.sparse._csc.csc_matrix'>


In [14]:
G_nx = nx.Graph()
G_nx.add_edges_from(edges)

In [17]:
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

In [19]:
diameter

8

In [22]:
aggregate_of, n_coarse = build_grouped_aggregation(G_nx, gamma_in, gamma_out, max_size=10)

In [25]:
summary = summarize_aggregation(aggregate_of, n_coarse, gamma_in, gamma_out)

n_coarse: 1990 (1990 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.03
Singletons: 40 (2.0%)
gamma_in_coarse: 365, gamma_out_coarse: 152
Overlap (must be empty): set()
Interior coarse vertices: 1473 (74.0%)


In [28]:
coarse_edges, coarse_contribs = build_coarse_graph_edges(edges, aggregate_of)
print(f"coarse edges: {len(coarse_edges)}")

coarse edges: 48947


In [32]:
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 755, gamma_out: 327, n_vertices: 4039
Boundary fraction: 26.7888%
  -> HIGH RISK. Comparable to or worse than confirmed failures (Rhesus 25%, bio-CE-GN 70%). Consider a different gamma-selection method, or (for diameter-endpoint selection) a smaller k_hop relative to the graph's diameter, before proceeding.
n_coarse: 1963 (1963 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.06
Singletons: 13 (0.7%)
gamma_in_coarse: 366, gamma_out_coarse: 216
Overlap (must be empty): set()
Interior coarse vertices: 1381 (70.4%)
coarse edges: 49230 (from 88234 fine edges)


In [35]:
result = run_paired_validation(setup, N=100)

Paired samples: 100%|██████████| 100/100 [01:14<00:00,  1.35sample/s, Q_fine=102.8006, Q_coarse=139.0798]


N = 100 paired samples
Q_fine   : mean=102.800580  var=2417.459982
Q_coarse : mean=139.079791  var=4431.763639
Q_fine - Q_coarse : mean=-36.279211  var=303.189840
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 7.97x


In [40]:
estimate = two_level_estimate(setup, result, N_coarse_only=300)

Coarse-only samples: 100%|██████████| 300/300 [01:16<00:00,  3.90sample/s, Q_coarse=140.0946]


Coarse-only base estimate (N=300): 140.094600
Correction term mean (paired samples): -36.279211
Two-level estimate of E[Q_fine]: 103.815389
Direct fine-only mean (for comparison): 102.800580


## Final run

In [45]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

G_nx = nx.Graph()
G_nx.add_edges_from(edges)

communities = list(greedy_modularity_communities(G_nx))
communities_sorted = sorted(communities, key=len, reverse=True)

print(f"Found {len(communities_sorted)} communities")
print(f"Sizes (top 5): {[len(c) for c in communities_sorted[:5]]}")

def top_degree_subset(community, G_nx, fraction=0.05):
    community_list = list(community)
    degrees = [(v, G_nx.degree(v)) for v in community_list]
    degrees_sorted = sorted(degrees, key=lambda x: x[1], reverse=True)
    n_keep = max(1, int(len(community_list) * fraction))
    return [v for v, d in degrees_sorted[:n_keep]]

gamma_in = top_degree_subset(communities_sorted[0], G_nx, fraction=0.05)
gamma_out = top_degree_subset(communities_sorted[1], G_nx, fraction=0.05)

print(f"gamma_in: {len(gamma_in)}, gamma_out: {len(gamma_out)}")
boundary_fraction = (len(gamma_in) + len(gamma_out)) / n_vertices
print(f"boundary fraction: {boundary_fraction:.4f}")

Found 13 communities
Sizes (top 5): [983, 815, 548, 543, 372]
gamma_in: 49, gamma_out: 40
boundary fraction: 0.0220


In [47]:
from sksparse.cholmod import cholesky as sparse_cholesky

setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 49, gamma_out: 40, n_vertices: 4039
Boundary fraction: 2.2035%
  -> Untested range -- proceed but verify interior_coarse is non-empty and substantial after aggregation.
n_coarse: 1967 (1967 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.05
Singletons: 13 (0.7%)
gamma_in_coarse: 49, gamma_out_coarse: 40
Overlap (must be empty): set()
Interior coarse vertices: 1878 (95.5%)
coarse edges: 48683 (from 88234 fine edges)


### Single-level MC

In [50]:
import time

t0 = time.time()
Q_samples = monte_carlo_loop_tqdm(L_sigma, lambda_min, edges, n_vertices,
                                    gamma_in, gamma_out, N=1000, debug=False)
single_level_time = time.time() - t0

single_level_se = Q_samples.std() / np.sqrt(len(Q_samples))

print(f"\n=== Single-Level MC on Facebook Ego ===")
print(f"Time: {single_level_time:.2f}s")
print(f"Mean Q: {Q_samples.mean():.6f}")
print(f"Std Q: {Q_samples.std():.6f}")
print(f"Standard error: {single_level_se:.6f}")

Monte Carlo: 100%|██████████| 1000/1000 [04:12<00:00,  3.96sample/s, mean Q=792.5822]


✓ Done — 1000 samples
  Mean Q : 792.582205
  Std Q  : 371.618320

=== Single-Level MC on Facebook Ego ===
Time: 253.42s
Mean Q: 792.582205
Std Q: 371.618320
Standard error: 11.751603


### Two-level MC Run

In [53]:
import time

t0 = time.time()
result = run_paired_validation(setup, N=100)
paired_time = time.time() - t0
print(f"Paired time (N=100): {paired_time:.2f}s")

Paired samples: 100%|██████████| 100/100 [00:47<00:00,  2.11sample/s, Q_fine=779.3359, Q_coarse=974.6409]


N = 100 paired samples
Q_fine   : mean=779.335904  var=140196.101244
Q_coarse : mean=974.640922  var=219256.545083
Q_fine - Q_coarse : mean=-195.305018  var=8802.403277
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 15.93x
Paired time (N=100): 47.45s


In [55]:
paired_diff_var = result['diff_samples'].var()
se_correction = np.sqrt(paired_diff_var / 100)
print(f"SE from correction term (N_paired=100): {se_correction:.6f}")

# solve for the coarse-only variance needed to hit the remaining budget
target_se = 11.751603
se_coarse_needed_sq = target_se**2 - se_correction**2

if se_coarse_needed_sq <= 0:
    print("Paired samples alone already meet or exceed target precision!")
else:
    se_coarse_needed = np.sqrt(se_coarse_needed_sq)
    print(f"SE needed from coarse-only term: {se_coarse_needed:.6f}")

SE from correction term (N_paired=100): 9.382112
SE needed from coarse-only term: 7.076450


In [57]:
# quick pilot run to estimate coarse-only variance
pilot_N = 100
Q_coarse_pilot = np.array([run_coarse_only_sample(setup, seed=n) for n in range(pilot_N)])
coarse_var_estimate = Q_coarse_pilot.var()
print(f"Pilot coarse-only variance estimate: {coarse_var_estimate:.4f}")

N_coarse_needed = coarse_var_estimate / se_coarse_needed**2
print(f"Estimated N_coarse_only needed: {int(np.ceil(N_coarse_needed))}")

Pilot coarse-only variance estimate: 219256.5451
Estimated N_coarse_only needed: 4379


In [59]:
import time
t0 = time.time()
Q_coarse_pilot_timed = np.array([run_coarse_only_sample(setup, seed=n) for n in range(100)])
pilot_time = time.time() - t0
print(f"100 coarse-only samples took: {pilot_time:.2f}s")
print(f"Estimated time for {4379} samples: {pilot_time * 4379 / 100:.2f}s ({pilot_time * 4379 / 100 / 60:.1f} min)")

100 coarse-only samples took: 44.84s
Estimated time for 4379 samples: 1963.52s (32.7 min)


In [ ]:
import time

N_coarse_only = 3000
t0 = time.time()
Q_coarse_only_samples = np.array([
    run_coarse_only_sample(setup, seed=100000+n) for n in range(N_coarse_only)
])
coarse_time = time.time() - t0
print(f"Coarse-only time (N={N_coarse_only}): {coarse_time:.2f}s")

# ---- combine into final two-level estimate and SE ----
coarse_base_mean = Q_coarse_only_samples.mean()
correction_mean = result['diff_samples'].mean()
two_level_estimate_value = coarse_base_mean + correction_mean

se_correction = np.sqrt(result['diff_samples'].var() / 100)
se_coarse = np.sqrt(Q_coarse_only_samples.var() / N_coarse_only)
combined_se = np.sqrt(se_correction**2 + se_coarse**2)

total_two_level_time = paired_time + coarse_time

print(f"\n=== Two-Level MC Summary (Facebook Ego) ===")
print(f"Two-level estimate: {two_level_estimate_value:.6f}")
print(f"Combined SE: {combined_se:.6f}")
print(f"Target SE: 11.751603")
print(f"Total time: {total_two_level_time:.2f}s")
print(f"Fine samples used: 100 (paired only)")